In [1]:
from infection_propagation import Graph
from infection_propagation import Figures
import importlib
import json
importlib.reload(Graph)
importlib.reload(Figures)

<module 'infection_propagation.Figures' from '/home/danny/cuboulder/in-progress/research-lladser/infection_propagation/src/infection_propagation/Figures.py'>

# Bidirectional Search Toy, Not Full, Implemented, Sometimes Returns Incorrect Path

In [2]:
threshold_value = lambda x : np.log(x)/x
n = 10
p = 1.2*threshold_value(n)
h = Graph.erdos_renyi(n, p, force_connection=True)
print(f"Threshold value: {threshold_value(n)}")
print(f"P value: {p}")

Threshold value: 0.23025850929940458
P value: 0.2763102111592855


In [3]:
h._adjency_matrix

,2,4,5,6,7,8,9,0,1,3
0,,,,,,,,,,
0,inf,<tree_...,<tree_...,<tree_...,inf,<tree_...,inf,inf,inf,inf
1,<tree_...,inf,inf,<tree_...,inf,inf,inf,inf,inf,inf
2,inf,inf,inf,inf,<tree_...,inf,<tree_...,inf,<tree_...,inf
3,inf,<tree_...,<tree_...,inf,<tree_...,inf,inf,inf,inf,inf
4,inf,inf,inf,inf,inf,<tree_...,<tree_...,<tree_...,inf,<tree_...
5,inf,inf,inf,inf,inf,inf,inf,<tree_...,inf,<tree_...
6,inf,inf,inf,inf,<tree_...,inf,inf,<tree_...,<tree_...,inf
7,<tree_...,inf,inf,<tree_...,inf,inf,<tree_...,inf,inf,<tree_...
8,inf,<tree_...,inf,inf,inf,inf,<tree_...,<tree_...,inf,inf


In [4]:
t = h.simulate_gossip_rv('1', '4')
print(t)

1.9485696158506007


In [5]:
h._adjency_matrix

,2,4,5,6,7,8,9,0,1,3
0,,,,,,,,,,
0,inf,0.218303,inf,inf,inf,1.69171,inf,inf,inf,inf
1,inf,inf,inf,inf,inf,inf,inf,inf,inf,inf
2,inf,inf,inf,inf,inf,inf,0.98908,inf,inf,inf
3,inf,inf,inf,inf,-0.118721,inf,inf,inf,inf,inf
4,inf,inf,inf,inf,inf,<tree_...,<tree_...,0.218303,inf,inf
5,inf,inf,inf,inf,inf,inf,inf,inf,inf,inf
6,inf,inf,inf,inf,inf,inf,inf,inf,inf,inf
7,inf,inf,inf,inf,inf,inf,1.441093,inf,inf,-0.118721
8,inf,<tree_...,inf,inf,inf,inf,<tree_...,1.69171,inf,inf


In [6]:
h.sim_all(reset=True)

In [7]:
h._adjency_matrix

,2,4,5,6,7,8,9,0,1,3
0,,,,,,,,,,
0,inf,1.39024,2.410061,0.251091,inf,0.041573,inf,inf,inf,inf
1,0.07047,inf,inf,0.082847,inf,inf,inf,inf,inf,inf
2,inf,inf,inf,inf,0.368002,inf,0.797963,inf,0.07047,inf
3,inf,2.686455,0.472039,inf,0.644611,inf,inf,inf,inf,inf
4,inf,inf,inf,inf,inf,2.262599,1.06489,1.39024,inf,2.686455
5,inf,inf,inf,inf,inf,inf,inf,2.410061,inf,0.472039
6,inf,inf,inf,inf,0.047173,inf,inf,0.251091,0.082847,inf
7,0.368002,inf,inf,0.047173,inf,inf,0.177221,inf,inf,0.644611
8,inf,2.262599,inf,inf,inf,inf,1.399977,0.041573,inf,inf


In [8]:
t = h.algo_jump('1', '4', algorithm="")
print(t)

1.3721304077843057


In [9]:
h.construct_path('1', '4')

['4', '9', '7', '6', '1']

In [10]:
t = h.algo_jump('1', '4', algorithm="bidir")
print(t)

1.3721304077843057


In [11]:
h.construct_path('1','4')

['4', '9', '7', '6', '1']

In [12]:
# How many times will it return the same path?
def accuracy_trial(iters=10**3, edgedst=None):
    blind_times = []
    blind_paths = []
    informed_times = []
    informed_paths = []
    for i in range(iters):
        h = Graph.erdos_renyi(n, p, force_connection=True, edge_dst=edgedst)
        h.sim_all()
        blind_times.append(h.algo_jump('1', '4', algorithm="bidir"))
        p1 = h.construct_path('1','4')
        blind_paths.append(p1)
        informed_times.append(h.algo_jump('1', '4', algorithm=""))
        p2 = h.construct_path('1','4')
        informed_paths.append(p2)
    return blind_times, blind_paths, informed_times, informed_paths

In [13]:
a,b,c,d = accuracy_trial()

# Look at non reversible RV, like a Uniform RV

# Do we need to add auxillary nodes to help the precision on the Dijsktra's path?
- ie we add more "fake observers" that connect to our src and dst subtrees to make a subgraph, and the shortest path we find in this subgraph may likely agree with the shortest path?

https://arxiv.org/pdf/1911.01521`
- first paragraph, note about adjacent matrix, $D(u,w) = D(u,v) - 1)$ or $D(u,w) = D(u,v) + 1$

First start advantage?
- how likely is the first minimal edge on the frontier on the final shortest path?

MOre results
- [X] Can we show dijsktra's shows a similar distribution? 
    - [X] Time dst? 
    - [X] Path Dst?
- [X] Bidirectional is good here right? Adding extra info about new delays doesn't change shortest path right?
    - Yes! Dijsktra's and Blind Search give shortest paths
- [X] At what amound of auxillary nodes is optimal for constraining the graph? How do we find these optimal auxillary nodes?
    - Bidirectional is instance optimal with nodes expanded $O(\Delta G)$, $\Delta G$ is maximal degree of graph

## Instance Optimal? Can not relax fewer edges on the graph asympotically, no need to find "auxillary nodes"

https://arxiv.org/abs/2410.14638
- bidirectional A* is more optimal but we'd need to beable to formulate a heuristic (very difficult)

## Trad Graph

In [14]:
threshold_value = lambda x : np.log(x)/x
n = 10
p = 1.2*threshold_value(n)
h = Graph.erdos_renyi(n, p, force_connection=True)
print(f"Threshold value: {threshold_value(n)}")
print(f"P value: {p}")

Threshold value: 0.23025850929940458
P value: 0.2763102111592855


In [15]:
h._adjency_matrix

,4,5,6,7,8,9,0,1,2,3
0,,,,,,,,,,
0,inf,inf,<tree_...,<tree_...,<tree_...,<tree_...,inf,inf,inf,inf
1,inf,<tree_...,inf,inf,inf,<tree_...,inf,inf,inf,inf
2,<tree_...,inf,<tree_...,inf,<tree_...,<tree_...,inf,inf,inf,inf
3,<tree_...,inf,inf,inf,inf,inf,inf,inf,inf,inf
4,inf,<tree_...,inf,<tree_...,<tree_...,inf,inf,inf,<tree_...,<tree_...
5,<tree_...,inf,inf,inf,inf,inf,inf,<tree_...,inf,inf
6,inf,inf,inf,inf,<tree_...,inf,<tree_...,inf,<tree_...,inf
7,<tree_...,inf,inf,inf,inf,<tree_...,<tree_...,inf,inf,inf
8,<tree_...,inf,<tree_...,inf,inf,<tree_...,<tree_...,inf,<tree_...,inf


In [16]:
t = h.simulate_gossip_rv('1', '4', preserve_times=True)
print(h.construct_path('1','4'))
print(t)

['4', '5', '1']
0.7448095222284945


In [17]:
h._adjency_matrix

,4,5,6,7,8,9,0,1,2,3
0,,,,,,,,,,
0,inf,inf,<tree_...,<tree_...,<tree_...,<tree_...,inf,inf,inf,inf
1,inf,0.128284,inf,inf,inf,1.094388,inf,inf,inf,inf
2,<tree_...,inf,<tree_...,inf,<tree_...,<tree_...,inf,inf,inf,inf
3,<tree_...,inf,inf,inf,inf,inf,inf,inf,inf,inf
4,inf,0.616526,inf,<tree_...,<tree_...,inf,inf,inf,<tree_...,<tree_...
5,0.616526,inf,inf,inf,inf,inf,inf,0.128284,inf,inf
6,inf,inf,inf,inf,<tree_...,inf,<tree_...,inf,<tree_...,inf
7,<tree_...,inf,inf,inf,inf,<tree_...,<tree_...,inf,inf,inf
8,<tree_...,inf,<tree_...,inf,inf,<tree_...,<tree_...,inf,<tree_...,inf


In [18]:
h.sim_all()
t = h.algo_jump('1', '4', algorithm="")
print(h.construct_path('1','4'))
print(t)

['4', '5', '1']
0.7448095222284945


In [19]:
h._adjency_matrix

,4,5,6,7,8,9,0,1,2,3
0,,,,,,,,,,
0,inf,inf,0.701789,0.195791,0.313861,0.183014,inf,inf,inf,inf
1,inf,0.128284,inf,inf,inf,1.094388,inf,inf,inf,inf
2,1.017619,inf,0.528023,inf,1.967763,0.404443,inf,inf,inf,inf
3,0.009326,inf,inf,inf,inf,inf,inf,inf,inf,inf
4,inf,0.616526,inf,0.805551,2.303117,inf,inf,inf,1.017619,0.009326
5,0.616526,inf,inf,inf,inf,inf,inf,0.128284,inf,inf
6,inf,inf,inf,inf,1.424738,inf,0.701789,inf,0.528023,inf
7,0.805551,inf,inf,inf,inf,0.20784,0.195791,inf,inf,inf
8,2.303117,inf,1.424738,inf,inf,0.414891,0.313861,inf,1.967763,inf


In [20]:
# How many times will it return the same path?
def accuracy_trial(iters=10**3, edgedst=None):
    blind_times = []
    blind_paths = []
    informed_times = []
    informed_paths = []
    for i in range(iters):
        h = Graph.erdos_renyi(n, p, force_connection=True, edge_dst=edgedst)
        blind_times.append(h.simulate_gossip_rv('1', '4', preserve_times=True))
        p1 = h.construct_path('1','4')
        blind_paths.append(p1)
        h.sim_all()
        informed_times.append(h.algo_jump('1', '4', algorithm=""))
        p2 = h.construct_path('1','4')
        informed_paths.append(p2)
    return blind_times, blind_paths, informed_times, informed_paths

In [22]:
a,b,c,d = accuracy_trial()

In [23]:
set(np.isclose(a,c))

{np.True_}

In [24]:
b == d

True

## Uniform RVs

In [25]:
edge_dst = None
with open('../test_graphs/unit1.json') as json_file:
    edge_dst = json.load(json_file)

In [26]:
a,b,c,d = accuracy_trial(edgedst=edge_dst)

In [27]:
set(np.isclose(a,c))

{np.True_}

In [28]:
b==d

True